# M3 — Entrenamiento de Keypoints de Ruedas y Conteo de Ejes (YOLO11-pose)
Basado en el notebook `keypoints_llantas_yolo_detectron2_v3.ipynb` del docente.
Esquema de 5 keypoints por rueda: `['centro', 'contacto', 'superior', 'izquierda', 'derecha']` (`kpt_shape: [5, 3]`, `flip_idx: [0, 1, 2, 4, 3]`).

In [ ]:
!nvidia-smi -L || echo "Activa GPU T4 en Entorno de ejecucion > Cambiar tipo de entorno"
!pip install -q ultralytics==8.3.107 roboflow==1.1.60 pyyaml pandas matplotlib

In [ ]:
import os, glob, json, shutil, random, yaml, cv2, numpy as np, pandas as pd, torch
from pathlib import Path
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
FLIP_IDX = [0, 1, 2, 4, 3]

UMBRAL_CONTENCION = 0.7
UMBRAL_CERCANA = 0.65
SEPARACION_EJE = 0.6
print('Dispositivo:', DEVICE, '| PyTorch:', torch.__version__)

## Parte A (Opcional) — Celda 8 mejorada: Auto-etiquetado de ruedas para subir a Roboflow
Sube >=200 fotos laterales / 3/4 de vehículos (incluyendo camiones de 2, 3 y 4+ ejes, buses, minibuses, autos y motos), genera el dataset pre-etiquetado y corrige las etiquetas en **Roboflow (Keypoint Detection)**.

In [ ]:
EJECUTAR_AUTOETIQUETADO = False  # Cambia a True si vas a pre-etiquetar un .zip de fotos crudas

def kps_de_mascara(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 12:
        return None
    pts = c.reshape(-1, 2).astype(float)
    if len(pts) >= 5:
        (cx, cy), _, _ = cv2.fitEllipse(c)
    else:
        cx, cy = pts.mean(0)
    contacto = pts[pts[:, 1].argmax()]
    superior = pts[pts[:, 1].argmin()]
    izquierda = pts[pts[:, 0].argmin()]
    derecha = pts[pts[:, 0].argmax()]
    k = np.array([[cx, cy], contacto, superior, izquierda, derecha])
    return np.concatenate([k, np.ones((5, 1))], axis=1)

if EJECUTAR_AUTOETIQUETADO:
    from google.colab import files
    RAW_IMGS = Path('/content/m3_imagenes')
    DS_PRE = Path('/content/dataset_llantas')
    shutil.rmtree(RAW_IMGS, ignore_errors=True)
    shutil.rmtree(DS_PRE, ignore_errors=True)
    RAW_IMGS.mkdir(parents=True)
    for fn in files.upload():
        if fn.lower().endswith('.zip'):
            !unzip -qo "$fn" -d /content/m3_imagenes
        else:
            shutil.move(fn, RAW_IMGS / fn)

    yoloe = YOLOE('yoloe-11l-seg.pt')
    prompts = ['wheel', 'tire', 'car wheel', 'truck wheel']
    yoloe.set_classes(prompts, yoloe.get_text_pe(prompts))

    for s in ('train', 'val'):
        (DS_PRE / 'images' / s).mkdir(parents=True, exist_ok=True)
        (DS_PRE / 'labels' / s).mkdir(parents=True, exist_ok=True)

    imgs = sorted(p for p in RAW_IMGS.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'))
    random.seed(0); random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * 0.2))
    total_llantas = 0

    for i, ruta in enumerate(imgs):
        split = 'val' if i < n_val else 'train'
        im = cv2.imread(str(ruta))
        if im is None:
            continue
        H, W = im.shape[:2]
        r = yoloe.predict(im, conf=0.15, imgsz=1024, device=DEVICE, agnostic_nms=True, verbose=False)[0]
        lineas = []
        if r.masks is not None and len(r.boxes):
            for box, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy):
                if len(poly) < 3:
                    continue
                m = np.zeros((H, W), np.uint8)
                cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                k = kps_de_mascara(m)
                if k is None:
                    continue
                x1, y1, x2, y2 = box
                x1, y1 = min(x1, k[:, 0].min()), min(y1, k[:, 1].min())
                x2, y2 = max(x2, k[:, 0].max()), max(y2, k[:, 1].max())
                vals = [(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
                for kx, ky, _ in k:
                    vals += [min(max(kx / W, 0), 1), min(max(ky / H, 0), 1), 2]
                lineas.append('0 ' + ' '.join(f'{v:.6f}' for v in vals))
                total_llantas += 1
        base = f'{i:05d}_{ruta.stem}'
        cv2.imwrite(str(DS_PRE / 'images' / split / f'{base}.jpg'), im)
        (DS_PRE / 'labels' / split / f'{base}.txt').write_text('\n'.join(lineas))

    (DS_PRE / 'data.yaml').write_text(
        f'path: {DS_PRE}\ntrain: images/train\nval: images/val\nkpt_shape: [5, 3]\nflip_idx: {FLIP_IDX}\nnames:\n  0: llanta\n'
    )
    !cd /content && zip -qr dataset_llantas_prelabel.zip dataset_llantas
    print('Llantas pre-etiquetadas:', total_llantas, '-> /content/dataset_llantas_prelabel.zip')

## Parte B — Descargar dataset de Keypoints revisado desde Roboflow

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Roboflow API Key: ')
WORKSPACE = input('Roboflow workspace slug: ').strip()
PROJECT = input('Roboflow project slug (M3 llantas keypoints): ').strip()
VERSION = int(input('Version del dataset (ej. 1): ').strip())

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov11', location='/content/dataset_m3')
DS_M3 = Path(dataset.location)

yaml_p = DS_M3 / 'data.yaml'
cfg = yaml.safe_load(yaml_p.read_text())
cfg['kpt_shape'] = [5, 3]
cfg['flip_idx'] = FLIP_IDX
yaml_p.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('data.yaml verificado:', cfg)

## Parte C — Entrenamiento YOLO11s-pose + Evaluación OKS y Error de Conteo de Ejes

In [ ]:
modelo_pose = YOLO('yolo11s-pose.pt')
modelo_pose.train(
    data=str(DS_M3 / 'data.yaml'),
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/pose',
    name='m3_llantas',
    exist_ok=True,
)

In [ ]:
BEST_M3 = Path('/content/runs/pose/m3_llantas/weights/best.pt')
pose_model = YOLO(str(BEST_M3))
val_pose = pose_model.val(data=str(DS_M3 / 'data.yaml'))

def altura_visible(k):
    return max(1.0, float(k[1, 1] - k[2, 1]))

def contar_ejes_desde_llantas(llantas_kps, box_vehiculo=None):
    if not llantas_kps:
        return None
    hmax = max(altura_visible(k) for k in llantas_kps)
    cercanas = [k for k in llantas_kps if altura_visible(k) >= UMBRAL_CERCANA * hmax]
    if box_vehiculo is not None and len(cercanas) == 2:
        x1, y1, x2, y2 = box_vehiculo
        prop = (x2 - x1) / max(1.0, y2 - y1)
        if prop < 1.6:
            angostas = all((k[4, 0] - k[3, 0]) / altura_visible(k) < 0.65 for k in cercanas)
            misma_h = abs(cercanas[0][1, 1] - cercanas[1][1, 1]) < 0.15 * hmax
            if angostas and misma_h:
                return None
    xs = sorted(float(k[0, 0]) for k in cercanas)
    diam = float(np.median([altura_visible(k) for k in cercanas]))
    ejes = 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * diam)
    return max(2, ejes)

metricas_m3 = {
    'modelo': 'yolo11s-pose',
    'kpt_shape': [5, 3],
    'keypoints': KPT_NOMBRES,
    'box_map50': round(float(val_pose.box.map50), 4),
    'pose_map50': round(float(val_pose.pose.map50), 4),
    'pose_map50_95': round(float(val_pose.pose.map), 4),
}

# Evaluacion opcional de error de ejes si hay un CSV en /content/ejes_val.csv (filename,ejes_reales)
ejes_csv = Path('/content/ejes_val.csv')
if ejes_csv.exists():
    df_e = pd.read_csv(ejes_csv)
    errores, aciertos = [], []
    for _, row in df_e.iterrows():
        img_p = next(DS_M3.rglob(row['filename']), None)
        if img_p is None:
            continue
        r = pose_model.predict(str(img_p), conf=0.25, device=DEVICE, verbose=False)[0]
        kps = []
        if r.keypoints is not None and len(r.boxes):
            xy = r.keypoints.xy.cpu().numpy()
            kc = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(xy.shape[:2])
            for p, c in zip(xy, kc):
                kps.append(np.concatenate([p, c[:, None]], 1))
        pred_ejes = contar_ejes_desde_llantas(kps) or 2
        real_ejes = int(row['ejes_reales'])
        errores.append(abs(pred_ejes - real_ejes))
        aciertos.append(int(pred_ejes == real_ejes))
    if errores:
        metricas_m3['ejes_mae'] = round(float(np.mean(errores)), 4)
        metricas_m3['ejes_exactitud'] = round(float(np.mean(aciertos)), 4)

print(json.dumps(metricas_m3, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m3.json').write_text(json.dumps(metricas_m3, indent=2, ensure_ascii=False))
shutil.copy(BEST_M3, '/content/best_m3_pose.pt')
print('Peso listo para RunPod: /content/best_m3_pose.pt')

In [ ]:
import sys, platform
print('Diagnostico Colab M3:', json.dumps({
    'python': sys.version.split()[0], 'os': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
from google.colab import files
files.download('/content/best_m3_pose.pt')
files.download('/content/resumen_metricas_m3.json')